## 01. Import dependancies

In [38]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder)

## 02. Load dataset

In [39]:
df = pd.read_csv('data/processed/binning_applied.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureBins
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,Newer
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,Medium
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,Newer
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,Long-term
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,Newer


## 03. Data preprocessing

In [40]:
ordinal_features = ['TenureBins'] # Ordinal encoder

#OneHot encoder
nominal_features = ['gender','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']
numerical_features = ['MonthlyCharges',	'TotalCharges']
remainder_features = ['SeniorCitizen']
target = 'Churn' # label encoder

## 04. Data splitting

In [41]:
X = df.drop(columns=[target])
y = df[target]

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [43]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (5625, 19)
y_train: (5625,)
X_test: (1407, 19)
y_test: (1407,)


## 05 Scaling and Encoding

In [44]:
from numpy import unique_values
numerical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

nominal_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='constant',
                fill_value='missing'
            )
        ),
        ('encoder', OneHotEncoder())
    ]
)

ordinal_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='constant',
                fill_value='missing'
            )
        ),
        ('encoder', OrdinalEncoder())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features),
    ],
    remainder='passthrough'
)

nominal_features_name=[]
for feature in nominal_features:
    unique_values=df[feature].unique()
    nominal_features_name.extend([f'{feature}_{val}' for val in unique_values])

In [45]:
X_train = pd.DataFrame(
    preprocessor.fit_transform(X_train,y_train),
    columns=numerical_features+nominal_features_name+ordinal_features+remainder_features
)

In [46]:
X_train

,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_Yes,Partner_No,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,...,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaperlessBilling_No,PaymentMethod_Electronic check,PaymentMethod_Mailed check,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),TenureBins,SeniorCitizen
0,0.981556,1.659900,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,-0.971546,-0.562252,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.837066,1.756104,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.641092,-0.908326,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0
4,-0.808787,-0.101561,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5620,0.197658,0.990321,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
5621,0.342148,-0.507338,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
5622,0.926749,0.473507,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
5623,0.029917,-0.721544,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,0.0


In [47]:
X_test = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=numerical_features+nominal_features_name+ordinal_features+remainder_features
)

In [48]:
X_test

,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_Yes,Partner_No,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,...,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaperlessBilling_No,PaymentMethod_Electronic check,PaymentMethod_Mailed check,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),TenureBins,SeniorCitizen
0,0.363738,0.984674,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.450100,-0.781798,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,2.0,0.0
2,-1.491376,-0.537223,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,-1.473107,-0.994619,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0
4,1.333645,2.308692,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,-0.664297,-0.781182,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,2.0,0.0
1403,-0.637725,-0.507536,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1404,-1.512966,-0.981192,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,0.0
1405,-0.812109,-0.400367,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [49]:
le = LabelEncoder()
y_train = pd.DataFrame(le.fit_transform(y_train), columns=['Churn'])
y_test = pd.DataFrame(le.transform(y_test), columns=['Churn'])

In [50]:
y_train

,Churn
0,0
1,0
2,0
3,0
4,0
...,...
5620,0
5621,1
5622,0
5623,0


In [51]:
y_test

,Churn
0,0
1,0
2,0
3,1
4,0
...,...
1402,0
1403,0
1404,0
1405,0


In [52]:
print(y_train['Churn'].value_counts())

Churn
0    4130
1    1495
Name: count, dtype: int64


## 06. Saving files

In [53]:
X_train.to_csv('artifacts/X_train.csv', index=False)
X_test.to_csv('artifacts/X_test.csv', index=False)
y_train.to_csv('artifacts/y_train.csv', index=False)
y_test.to_csv('artifacts/y_test.csv', index=False)